# 01 数据下载与项目初始化
本 Notebook 完成：
1. 自选 10 只 A 股 5 年后复权日度行情
2. 市场指数（沪深300、中证500）
3. 宏观经济指标（CPI 同比、M2 同比）
4. 近 5 年财务指标（ROE、净利润率）
5. 创建项目文件夹并按规范存储全部数据

环境：Python 3.11.15，`pip install baostock akshare pandas`

In [ ]:
import baostock as bs
import pandas as pd
from datetime import datetime
import time
import os
import re
import shutil

LOG_FILE = "download_log.txt"

def write_log(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {message}\n")
    print(message)

if os.path.exists(LOG_FILE):
    os.remove(LOG_FILE)
write_log("开始数据下载（完整项目版）")

In [ ]:
lg = bs.login()
print('login respond error_code:', lg.error_code)
print('login respond error_msg:', lg.error_msg)
write_log(f"Baostock 登录状态: {lg.error_code} - {lg.error_msg}")

In [ ]:
stocks = [
    ("sh.601398", "工商银行", "银行"),
    ("sh.600036", "招商银行", "银行"),
    ("sz.002594", "比亚迪", "汽车"),
    ("sh.600104", "上汽集团", "汽车"),
    ("sz.000002", "万科A", "房地产"),
    ("sh.600048", "保利发展", "房地产"),
    ("sh.600519", "贵州茅台", "白酒"),
    ("sz.000858", "五粮液", "白酒"),
    ("sh.601088", "中国神华", "能源"),
    ("sh.600900", "长江电力", "能源"),
]
print(f"已定义 {len(stocks)} 只股票")

## 1. 股票日线行情（后复权）

In [ ]:
stock_data = {}
start_date = "2020-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")

for bs_code, name, industry in stocks:
    try:
        rs = bs.query_history_k_data_plus(
            bs_code,
            "date,open,close,high,low,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="2"
        )
        if rs.error_code != '0':
            raise Exception(rs.error_msg)
        data_list = []
        while rs.next():
            data_list.append(rs.get_row_data())
        df = pd.DataFrame(data_list, columns=rs.fields)
        df = df.rename(columns={
            "date": "日期", "open": "开盘", "close": "收盘",
            "high": "最高", "low": "最低", "volume": "成交量", "amount": "成交额"
        })
        stock_data[bs_code] = df
        write_log(f"SUCCESS  stock_{bs_code}  shape={df.shape}")
        time.sleep(0.5)
    except Exception as e:
        write_log(f"FAILED   stock_{bs_code}  Error: {str(e)}")

print(f"\n成功下载 {len(stock_data)} 只股票")

## 2. 市场指数

In [ ]:
index_data = {}
index_list = [("sh.000300", "沪深300"), ("sh.000905", "中证500")]

for idx_code, idx_name in index_list:
    try:
        rs = bs.query_history_k_data_plus(
            idx_code,
            "date,open,close,high,low,volume,amount",
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag="2"
        )
        if rs.error_code != '0':
            raise Exception(rs.error_msg)
        data_list = []
        while rs.next():
            data_list.append(rs.get_row_data())
        df_idx = pd.DataFrame(data_list, columns=rs.fields)
        index_data[idx_code] = df_idx
        write_log(f"SUCCESS  index_{idx_code} ({idx_name})  shape={df_idx.shape}")
    except Exception as e:
        write_log(f"FAILED   index_{idx_code} ({idx_name})  Error: {str(e)}")

print(f"\n成功下载指数数量：{len(index_data)}")

## 3. 财务指标（ROE、净利润率）

In [ ]:
financial_records = []
years = [2020, 2021, 2022, 2023, 2024]

for bs_code, name, industry in stocks:
    try:
        for y in years:
            rs_profit = bs.query_profit_data(code=bs_code, year=y, quarter=4)
            if rs_profit.error_code == '0':
                data_list = []
                while rs_profit.next():
                    data_list.append(rs_profit.get_row_data())
                if data_list:
                    df = pd.DataFrame(data_list, columns=rs_profit.fields)
                    if 'roeAvg' in df.columns:
                        roe_val = pd.to_numeric(df['roeAvg'].iloc[0], errors='coerce')
                        if pd.notna(roe_val):
                            financial_records.append((bs_code, y, 'ROE', float(roe_val)))
                    if 'npMargin' in df.columns:
                        npm_val = pd.to_numeric(df['npMargin'].iloc[0], errors='coerce')
                        if pd.notna(npm_val):
                            financial_records.append((bs_code, y, '净利润率', float(npm_val)))
            else:
                write_log(f"WARNING  profit {bs_code} {y} {rs_profit.error_msg}")
            time.sleep(0.3)
        write_log(f"SUCCESS  fin_{bs_code} processed")
    except Exception as e:
        write_log(f"FAILED   fin_{bs_code} Error: {str(e)}")

fin_long = pd.DataFrame(financial_records, columns=["code", "year", "indicator", "value"])
print(f"\n财务数据长格式维度：{fin_long.shape}")
write_log(f"SUMMARY  financial_long_format  shape={fin_long.shape}")

## 4. 宏观经济指标

In [ ]:
macro_data = {}
try:
    import akshare as ak
except ImportError:
    write_log("WARNING  akshare 未安装，跳过宏观指标")
    ak = None

def parse_cn_month(month_str):
    if pd.isna(month_str):
        return None
    month_str = str(month_str).strip()
    match = re.search(r'(\d{4})\D*(\d{1,2})', month_str)
    if match:
        return f"{match.group(1)}-{match.group(2).zfill(2)}"
    return None

if ak:
    # CPI
    try:
        cpi_raw = ak.macro_china_cpi_monthly()
        if '商品' in cpi_raw.columns:
            pattern = '居民消费价格|CPI|消费者物价|全国CPI|同比'
            mask = cpi_raw['商品'].str.contains(pattern, na=False, regex=True)
            cpi_df = cpi_raw[mask].copy()
            if not cpi_df.empty:
                cpi_df['date'] = cpi_df['日期'].apply(parse_cn_month)
                cpi_df['cpi_yoy'] = pd.to_numeric(cpi_df['今值'], errors='coerce')
                cpi_df = cpi_df.dropna(subset=['date', 'cpi_yoy'])
                cpi_df = cpi_df[cpi_df['date'] >= '2020-01']
                macro_data['cpi'] = cpi_df[['date', 'cpi_yoy']]
                write_log(f"SUCCESS  macro_cpi  shape={cpi_df.shape}")
            else:
                available = cpi_raw['商品'].unique().tolist()
                raise Exception(f"未匹配到CPI行，可用商品: {available}")
        else:
            raise Exception(f"未找到‘商品’列，可用列: {list(cpi_raw.columns)}")
    except Exception as e:
        write_log(f"FAILED   macro_cpi  Error: {str(e)}")

    # M2
    try:
        m2_raw = ak.macro_china_money_supply()
        date_col = None
        m2_col = None
        for col in m2_raw.columns:
            if '月份' in col or '日期' in col or 'date' in col.lower():
                date_col = col
            if 'm2' in col.lower() and ('同比' in col or 'yoy' in col.lower()):
                m2_col = col
        if date_col is None or m2_col is None:
            date_col = '月份' if '月份' in m2_raw.columns else m2_raw.columns[0]
            m2_col = 'M2同比' if 'M2同比' in m2_raw.columns else None
        if m2_col is None:
            raise KeyError(f"找不到M2同比列，现有列: {list(m2_raw.columns)}")
        m2_df = m2_raw[[date_col, m2_col]].copy()
        m2_df['date'] = m2_df[date_col].apply(parse_cn_month)
        m2_df = m2_df.dropna(subset=['date'])
        m2_df = m2_df.rename(columns={m2_col: 'm2_yoy'})
        m2_df = m2_df[m2_df['date'] >= '2020-01']
        macro_data['m2'] = m2_df[['date', 'm2_yoy']]
        write_log(f"SUCCESS  macro_m2  shape={m2_df.shape}")
    except Exception as e:
        write_log(f"FAILED   macro_m2  Error: {str(e)}")
else:
    write_log("WARNING  跳过宏观指标下载")

print(f"\n成功下载宏观指标数量：{len(macro_data)}")

## 5. 构建项目文件夹并存储数据

In [ ]:
PROJECT_ROOT = "dshw-p01"

# 创建所有子目录
dirs = [
    "data/stock",
    "data/index",
    "data/macro",
    "data/finance",
    "data/clean",
    "data/combined",
    "output"
]
for d in dirs:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)

# 保存股票行情
for bs_code, df in stock_data.items():
    code_6 = bs_code.split(".")[1]
    fname = f"stock_{code_6}.csv"
    df.to_csv(os.path.join(PROJECT_ROOT, "data/stock", fname), index=False)

# 保存指数
for idx_code, df in index_data.items():
    code_6 = idx_code.split(".")[1]
    fname = f"index_{code_6}.csv"
    df.to_csv(os.path.join(PROJECT_ROOT, "data/index", fname), index=False)

# 保存宏观
for key, df in macro_data.items():
    fname = f"macro_{key}.csv"
    df.to_csv(os.path.join(PROJECT_ROOT, "data/macro", fname), index=False)

# 保存财务
fin_long.to_csv(os.path.join(PROJECT_ROOT, "data/finance", "finance_ratios.csv"), index=False)

# 复制日志
if os.path.exists(LOG_FILE):
    shutil.copy(LOG_FILE, os.path.join(PROJECT_ROOT, LOG_FILE))

# 生成 .gitignore 和 requirements.txt
with open(os.path.join(PROJECT_ROOT, ".gitignore"), "w") as f:
    f.write("data/\noutput/\n*.pyc\n.ipynb_checkpoints/\ndownload_log.txt\n")
with open(os.path.join(PROJECT_ROOT, "requirements.txt"), "w") as f:
    f.write("baostock\nakshare\npandas\n")

write_log("项目文件夹构建完成，所有数据已保存至 dshw-p01/")
print("\n✅ 数据下载与项目初始化全部完成！")

## 6. 退出 Baostock 并查看日志

In [ ]:
bs.logout()
print("\n===== 下载日志（最后10行） =====")
with open(LOG_FILE, "r", encoding="utf-8") as f:
    lines = f.readlines()
    for line in lines[-10:]:
        print(line, end='')